# Gaussian states and gates

Objectives:  
1. Understand as a minimum equations (20) and (22) in Jonatan's reference, as well as those in gray boxes in section III. The covariance matrices and displacement vectors of the four classes of single-mode Gaussian states mentioned above are presented in section VI. We will not consider two-mode or multi-mode states in this exercise.
2. Create functions for calculating the Wigner function of a general Gaussian state.
3. Using contour plot, 3D surface plot, or similar, plot the Wigner functions of various single-mode Gaussian states based on the formulas indicated above.\
Don't spend too much time on optimising the visuals of the plots in this exercise - you can go with the defaults, if you wish. But decide on a fixed visual style that you will maintain for all your plots.
4. Now, try out `interact` and/or `FuncAnimation` for varying parameters of your state plots (for example the coherent state amplitude or the degree of squeezing).
5. Write functions for performing Gaussian operations on your Gaussian states: displacement, phase shift and squeezing.
6. Your overall task is now to write a notebook, introducing Gaussian states and operations to your fellow students in a narrative style with text mixed with code and visualisations. The focus here should be on the physics and the visual explanation of it, not so much on discussing the code.
7. At the end of the day, share your notebook – however incomplete – on Discord.

## 1. Undstanding the equations
The reference is available in [Jonathan's reference](https://arxiv.org/pdf/2102.05748).

### 1.1 Equaitons (20) and (22)
$$
\begin{align}
    W(\vec{r}) &= \dfrac{1}{(2\pi)^n\sqrt{\det(\bm{\sigma})}}e^{-\frac{1}{2}(\vec{r}-\bar{r})^T\bm{\sigma}^{-1}(\vec{r}-\bar{r})} \tag{20} \\

    \bm{\sigma} \rightarrow \bm{F}\bm{\sigma}\bm{F}^T, \quad &\mathrm{and}, \quad \vec{r} \rightarrow \bm{F}\vec{r} + \vec{d} \tag{22}
\end{align}
$$

Eq. (20) has the form of the multivariate normal distribution, a gaussian in 2 or more dimensions (see the wiki-page: [multivariate normal distrubition](https://en.wikipedia.org/wiki/Multivariate_normal_distribution#Density_function)).  
<!-- It is implicit that $\vec{r} = (\vec{x},\vec{p})$, which in turn  -->
The bar, $\bar{r}$, indicates the mean position and sigma, $\sigma$, is the covariance-matrix (How the probability varies align one dimension in relation to another).
The paramters $n$ is the dimensionality of the vector $\vec{r} \in \mathbb{R}^n$ and the matrix $\bm{\sigma} \in \mathbb{R}^{n\times n}$

- NOTE: The difference in the exponent for $(2\pi)$ between Jonathan's reference and the Wiki definition, is due to $\vec{r} = (\vec{x},\vec{p})$ being implicit, and if each has $n$ demensions, the total is then $2n$, and where the term usually is represented in the squareroot, it can be factored out as shown above.

---
Eq. (22) explains how a guassian (unitary) transformation changes the covariance matrix and the displacement vector, $(\bm{\sigma},\vec{r})$

### 1.2 Some relevant equations in section III. Gaussian Unitaries

#### Displacement operation
$$\begin{align}
    \bm{F} &= \mathbb{I}_{2n} \tag{28} \\
    \vec{d} &= \sqrt{2}\begin{pmatrix} \mathrm{Re}(\alpha_1) \\ \mathrm{Im}(\alpha_1) \\ \vdots \\ \mathrm{Re}(\alpha_n) \\ \mathrm{Im}(\alpha_n)\end{pmatrix}
\end{align}$$
For a displacement of $\vec{\alpha}$

- This means that a displacement operator on the Wigner function does not alter the covariance matrix, but shifts the $\vec{r}$ as shown above.

#### Phase shift operation
$$\begin{align}
    \bm{F} &=
        \begin{pmatrix}
            \cos(\phi) & \sin(\phi) \\
            -\sin(\phi) & \cos(\phi) 
        \end{pmatrix} \equiv \bm{R}(\phi) \tag{33} \\

    \vec{d} &= \vec{0}
\end{align}$$

A phase shift of a Wigner function correpsonds to a rotation of the covariance matrix, and leaves the displacement unchanged.

#### Beam splitter
$$\begin{align}
    \bm{F} &= 
        \begin{bmatrix} 
        \sqrt{\eta}\ \mathbb{I}_{2} & \sqrt{1-\eta}\ \mathbb{I}_{2} \\
        \sqrt{1-\eta}\ \mathbb{I}_{2} & \sqrt{\eta}\ \mathbb{I}_{2} 
        \end{bmatrix} \equiv \bm{F}_{\eta} \tag{41} \\
        \vec{d} &= \vec{0} \tag{42}
\end{align}$$
Using a beamsplitter leaves the displacement at zero, and mixes the mode operators (shown is the two-mode beam-splitter).  
$\eta$ is the transmitivity.

## 2. Create function for generating Wigner functions for Gaussian states

In [ ]:
import numpy as np
from functools import reduce


class Wigner():
    
    def __init__(self, mean, sigma):
        self.state = {
            'mean': None,
            'cov': None,
        }
        self.mean = mean
        self.sigma = sigma
        
        self.dim = len(self.mean)
        self.__setattr__("mean", mean)
        self.__setattr__("cov", sigma)
        
        self.sigmaDet = np.linalg.det(self.sigma)
        self.sigmaInv = np.linalg.inv(self.sigma)
        self.norm = 1/((2*np.pi)**self.dim * np.sqrt(self.sigmaDet))
    
            
    def __call__(self, r, kwargs=None):
        if kwargs is not None:
            if np.any(["mean", "cov"]) in kwargs:
                if "mean" in kwargs:
                    self.mean = kwargs["mean"]
                if "cov" in kwargs:
                    self.sigma = kwargs["cov"]
            else:
                raise ValueError("kwargs must contain 'mean' or 'cov' to update the state.")
        
        r = np.array(r)
        assert r.shape == self.mean.shape, "Input point must match mean's shape"
        diff = r - self.mean
        
        # exponent = -0.5 * np.einsum("...j,jk,kl->l...", diff.T , self.sigmaInv , diff)
        exponent = -0.5 * diff.T @ self.sigmaInv @ diff
        
        return self.norm * np.exp(exponent)
    
    

x = np.array([0,0])[:, None]
p = np.array([1, 1])[:, None]

rbar = np.concatenate((x,p), axis=1).flatten()
sigma = np.eye(4)
sigma = sigma / np.linalg.norm(sigma)

test = Wigner(rbar, sigma)


np.float64(0.0004978489179803034)

{'mean': array([0, 1]),
 'cov': array([[0.5, 0. ],
        [0. , 0.5]])}